In [46]:
# Imports

import os
import sys
import pandas as pd
from scipy.stats import uniform , randint
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score , accuracy_score

sys.path.append(os.path.join(".."))
sys.path.append(os.path.join("../src"))

from preprocessing import get_X_y , get_X

In [4]:
# Data Load

df_train = pd.read_csv("../dataset/train.csv")
df_test = pd.read_csv("../dataset/test.csv")

df_train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [8]:
X , y = get_X_y(df_train)

In [112]:
X_train , X_val , y_train , y_val = train_test_split(
    X ,
    y ,
    test_size = 0.1 ,
    random_state = 7
)

cat_cols = X_train.select_dtypes(include = ["object" , "string"]).columns
num_cols = X_train.select_dtypes(include = ["number"]).columns

In [61]:
# 1st Try

preprocess = ColumnTransformer([
    ("cat" , OneHotEncoder(handle_unknown= "ignore") , cat_cols)
] , remainder = "passthrough"
)

pipeline = Pipeline([
    ("preprcoess" , preprocess) ,
    ("xgbc" , XGBClassifier(
        n_jobs = 1 ,
        subsample = 0.8 ,
        colsample_bytree = 0.8 ,
        random_state = 9
    ))
])

param = {
    "xgbc__n_estimators" : randint(200 , 401) ,
    "xgbc__max_depth" : randint(4 , 9),
    "xgbc__min_child_weight" : [1,2,3,4,5] ,
    "xgbc__gamma" : uniform(0 , 5) ,
    "xgbc__reg_lambda" : uniform(0 , 30) ,
    "xgbc__reg_alpha" : uniform(0 , 10) ,
    "xgbc__learning_rate" : uniform(0.01 , 0.29)
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param ,
    cv = 5 ,
    n_iter = 15 ,
    n_jobs = -1
)

srch.fit(X_train , y_train)

model = srch.best_estimator_

y_pred = model.predict(X_val)

f1 = f1_score(y_val , y_pred)
ac = accuracy_score(y_val , y_pred)

print(srch.best_params_)
print()
print(f"F1 Score : {f1}")
print(f"Accuracy Score : {ac}")

{'xgbc__gamma': np.float64(1.4912548919872322), 'xgbc__learning_rate': np.float64(0.031786849894729914), 'xgbc__max_depth': 7, 'xgbc__min_child_weight': 3, 'xgbc__n_estimators': 325, 'xgbc__reg_alpha': np.float64(2.287597736101039), 'xgbc__reg_lambda': np.float64(5.2335946459231355)}

F1 Score : 0.8399122807017544
Accuracy Score : 0.830034924330617


# Conidering max_depth = 7 , min_child_weight = 3 , n_estimators = 325 as fixed base line , after running this cell multiple times

In [94]:
# 2nd Try

preprocess = ColumnTransformer([
    ("cat" , OneHotEncoder(handle_unknown= "ignore") , cat_cols)
] , remainder = "passthrough"
)

pipeline = Pipeline([
    ("preprcoess" , preprocess) ,
    ("xgbc" , XGBClassifier(
        n_jobs = 1 ,
        subsample = 0.8 ,
        colsample_bytree = 0.8 ,
        random_state = 9,
        min_child_weight = 3 ,
        n_estimators = 325 ,
        max_depth = 7 
    ))
])

param = {
    "xgbc__gamma" : uniform(0 , 3) ,
    "xgbc__reg_lambda" : uniform(3 , 6) ,
    "xgbc__reg_alpha" : uniform(1 , 3) ,
    "xgbc__learning_rate" : uniform(0.02 , 0.03)
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param ,
    cv = 5 ,
    n_iter = 15 ,
    n_jobs = -1
)

srch.fit(X_train , y_train)

model = srch.best_estimator_

y_pred = model.predict(X_val)

f1 = f1_score(y_val , y_pred)
ac = accuracy_score(y_val , y_pred)

print(srch.best_params_)
print()
print(f"F1 Score : {f1}")
print(f"Accuracy Score : {ac}")

{'xgbc__gamma': np.float64(0.26245752724033977), 'xgbc__learning_rate': np.float64(0.045549787842080254), 'xgbc__reg_alpha': np.float64(2.288167967360052), 'xgbc__reg_lambda': np.float64(4.59542717492711)}

F1 Score : 0.8408342480790341
Accuracy Score : 0.8311990686845169


# Final Verdict
# -----------------------------------------------------------
# max_depth = 7
# min_child_weight = 3 
# n_estimators = 325
# gamma = 0.2624
# learning_rate = 0.0455
# reg_lambda = 4.5954
# reg_alpha = 2.2881